### In this notebook, we will learn the basics of Neural Nets from building out single neuron to MLP (Multi Layer Perceptrons)

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import random

In [6]:
class Value:
    def __init__(self, data, _children =(), _op = '' ):
        self.data = data
        self.grad = 0
        self._prev = set(_children)
        self._backward = lambda : None
        self._op = _op
    
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
            
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))

        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        
        out._backward = _backward
        return out
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rmul__(self, other):
        return self * other
    
    def __radd__(self, other):  
        return self + other
    
    def __rsub__(self, other):
        return other + (-self)
    
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        return self * other**-1
    
    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def exp(self):
        n = self.data 
        out = Value(math.exp(n), (self,))

        def _backward():
            self.grad += out.data * out.grad
        
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)

                for child in v._prev:
                    build_topo(child)

                topo.append(v)

        build_topo(self)

        self.grad = 1.0 # Because this is the output node and we will start the back propagation from here.

        for node in reversed(topo):
            node._backward()

In [3]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for i in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

In [4]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

In [5]:
class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x

In [6]:
x = [2.0, 3.0, -1.0]
n = Neuron(3)
out = n(x)
print(out)

Value(data=0.9551852782010036)


In [7]:
x = [2.0, 3.0, -1.0]
n = Layer(3, 3)
out = n(x)
print(out)

[Value(data=0.9592482042095771), Value(data=0.9998774299230027), Value(data=-0.40075069633565374)]


In [23]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4,4,1])
n(x)

Value(data=0.3250899116205883)

In [24]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, -1.0]

In [25]:
ypred = [n(x) for x in xs]
ypred

[Value(data=0.3250899116205883),
 Value(data=0.26035936684969035),
 Value(data=0.4308933280540761),
 Value(data=0.1891048475642594)]

### Now we can see that the error is high because the input values xs are not near to the target value ys. So we will now implement MSE loss function and try to optimize our Neural Nets using that loss function

In [26]:
loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])

We calculated the loss wrt out ground truth ys and we have squared it so that regardless of the negative or positive sign we always have the loss in positive.

In [27]:
loss.data

5.505435415772548

### Now we want to minimize this high loss because we want our neural network to learn better so that it can generalize to unseen examples

In [13]:
loss.backward()

In [14]:
n.layers[0].neurons[0].w[0]

Value(data=-0.2401477502513587)

In [15]:
n.layers[0].neurons[0].w[0].grad

-0.49587390520991087

### This is very important, as soon as we calculated the backward function, we calculated the loss. This means that now we have the gradients for all of the neurons and we now know that how much influence does each one of the neurons has on the output. 

n.layers[0].neurons[0].w[0].grad means the first layer's first neuron and we are seeing its first weight and .grad means that we are seeing its gradient so we now get the gradient value as -0.086 which means that if we slightly increase this value then if will have a -0.086 times influence on the output.

In [16]:
loss

Value(data=2.684989746043974)

### Now we will update our Neuron class to make sure we minimize the loss

In [2]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for i in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]

In [3]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

In [4]:
class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layers in self.layers for p in layers.parameters()]

### Now when we ran the for loop on top of data and again calculated new predictions, we could see that we have reduced the loss. Now we just need to iterate this process to minimize the loss.

Now lets run a training loop to minimize the loss as we are updating the loss using gradient descent

In [7]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4,4,1])
n(x)

Value(data=0.6920460676334588)

In [8]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, -1.0]  # desired targets / ground truth

In [10]:
for k in range(20):

    ## Forward Pass

    ypred = [n(x) for x in xs]
    loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])

    ## Backward Pass
    for p in n.parameters():
        p.grad = 0.0 # set the gradients to zero before backward pass
    loss.backward()

    ## Update Weights

    for p in n.parameters():
        p.data += -0.07 * p.grad

    print(k, loss.data)

0 9.184211461490147
1 4.685582550967579
2 2.9794260319116415
3 2.072665607460706
4 2.602257689900869
5 2.471009627743032
6 1.8225225492502912
7 1.4807291783488585
8 1.2811778882966682
9 1.3590995184286543
10 1.550240554260777
11 2.2939935222248007
12 1.077301002861879
13 0.7116056748442215
14 0.6712078732561698
15 0.5510085922698773
16 0.5025587771547961
17 0.31559830016965723
18 0.167009635796673
19 0.10682713033089641


In [11]:
ypred

[Value(data=0.7319151062024006),
 Value(data=-0.9564343993946294),
 Value(data=-0.9216033179243622),
 Value(data=-0.8359462931480943)]

### Now we can see that the values of ypred are very very close to the ground truth values as we had expected this means out gradient descent algorithm is working properly.